# Step 2 - Select one good and one bad human move per game

**Goal:** from every filtered game pick (at most) **one good move** and **one bad move** that the human actually
played, using Lichess' own evaluations as a pre-filter. Step 3 re-checks them with Stockfish.

**Input:** `outputs/step1_filter_games/games_filtered.pgn`
**Output** (`outputs/step2_select_good_bad_moves/`): `candidates.jsonl`, `counts.json`

**Rules** (defined once in `config.py` / `move_rules.py`, so steps 2 and 3 cannot disagree):
- position before the move is between full moves 10 and 30
- |evaluation before| <= 500 cp, no mate score before or after (not a decided game)
- **good** = the move loses <= 30 cp; **bad** = it loses >= 100 cp (or a win-probability drop, see `METRIC` in `config.py`)
- the side to move is not in check, there are >= 2 legal moves, and a position is never used twice
- among the eligible moves of a game, one per label is chosen by a seeded shuffle (`SEED + game number`) -> reproducible

**Key question for your professor:** how many games contain a bad move? He suggested adding bad moves if there are
about 40 or more. The counts are printed at the end.

In [ ]:
import json, random
from pathlib import Path
import chess, chess.pgn
import pandas as pd
import config as C
from move_rules import classify
from common import sha256_file, write_manifest

IN = C.OUT_DIR / "step1_filter_games" / "games_filtered.pgn"
OUT = C.OUT_DIR / "step2_select_good_bad_moves"
OUT.mkdir(parents=True, exist_ok=True)

## Find all eligible moves per game
Lichess stores the evaluation of the position *after* each move, so the evaluation *before* a move is the one stored on
the previous move. All values are converted to the mover's point of view.

In [ ]:
def eligible_moves(game):
    pools = {"good": [], "bad": []}
    node = game
    while node.variations:
        nxt = node.variation(0)
        board = node.board()
        pe, ne = node.eval(), nxt.eval()
        if pe and ne and C.FULLMOVE_MIN <= board.fullmove_number <= C.FULLMOVE_MAX:
            if not (pe.is_mate() or ne.is_mate()) and abs(pe.white().score()) <= C.MAX_ABS_EVAL_CP:
                mover = board.turn
                label, cp_loss, wp_loss = classify(pe.pov(mover).score(), ne.pov(mover).score())
                if label:
                    pools[label].append(dict(fen_before=board.fen(), move_uci=nxt.move.uci(),
                                             move_san=board.san(nxt.move), fullmove=board.fullmove_number,
                                             lichess_cp_loss=round(cp_loss), lichess_wp_loss=round(wp_loss, 1),
                                             lichess_eval_before_cp=pe.white().score()))
        node = nxt
    return pools

## Choose one move per label and game
The same position key (board, side, castling, en passant) is never used twice - not even between good and bad.

In [ ]:
rows, used, stats = [], set(), dict(games=0, with_good=0, with_bad=0, with_both=0, eligible_good_moves=0, eligible_bad_moves=0)
with open(IN, encoding="utf-8") as f:
    gi = 0
    while (game := chess.pgn.read_game(f)) is not None:
        gi += 1
        gid = game.headers.get("Site", f"game{gi}").rsplit("/", 1)[-1]
        pools = eligible_moves(game)
        stats["games"] += 1
        stats["eligible_good_moves"] += len(pools["good"]); stats["eligible_bad_moves"] += len(pools["bad"])
        got = {}
        for label in ("good", "bad"):
            pool = pools[label][:]
            random.Random(C.SEED + gi).shuffle(pool)
            for c in pool:
                b = chess.Board(c["fen_before"]); key = " ".join(c["fen_before"].split()[:4])
                if b.is_check() or b.legal_moves.count() < 2 or key in used:
                    continue
                used.add(key)
                got[label] = dict(cand_id=f"{gid}_{label[0]}", game_id=gid, label=label,
                                  white=game.headers.get("White"), black=game.headers.get("Black"),
                                  white_elo=game.headers.get("WhiteElo"), black_elo=game.headers.get("BlackElo"),
                                  time_control=game.headers.get("TimeControl"), **c)
                break
        stats["with_good"] += "good" in got; stats["with_bad"] += "bad" in got; stats["with_both"] += len(got) == 2
        rows += got.values()

with open(OUT / "candidates.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
stats["candidates_written"] = len(rows)
(OUT / "counts.json").write_text(json.dumps(stats, indent=2))
write_manifest(OUT / "manifest.json", "step2_select_good_bad_moves",
               {"input_sha256": sha256_file(IN), "output_sha256": sha256_file(OUT / "candidates.jsonl"), **stats})
stats

## How many bad moves are there?
`with_bad` is the number your professor asked for. At this rating level bad moves are rare, so this can be small.
If it is below ~40, either loosen the bad-move threshold (e.g. `METRIC = "wp"` or `BAD_MIN_CP_LOSS = 75` in `config.py`,
then re-run) or keep bad moves out of the main analysis.

In [ ]:
df = pd.DataFrame(rows)
print(f"games: {stats['games']} | with a good move: {stats['with_good']} | with a bad move: {stats['with_bad']} | both: {stats['with_both']}")
print("=> add bad moves (>= 40)?", "YES" if stats["with_bad"] >= 40 else "NOT ENOUGH - see note above")
df.groupby("label")[["lichess_cp_loss", "lichess_wp_loss", "fullmove"]].describe().T